<b><h1 style="font-size:30px;text-align: center;">- Podatkovno inženjerstvo - Vježba 5 -</h1></b>
<b><h1 style="font-size:25px;text-align: center;">- Dimenzijsko modeliranje u praksi: star shema, snowflake i SCD -</h1></b>

<div style="text-align:center; margin:16px 0 28px;">
  <img
    src="https://www.debug.hr/wp-content/uploads/2025/01/084_2.jpg"
    alt="Algebra Bernays"
    style="width:600px; max-width:60%; height:auto; margin-top:25px;"
  >
</div>

---------------------

<b>*Izrađeno: studeni 2026.*</b>

<b>*Autor: Mislav Spajić, mag. ing. comp.*</b>

---------------------

# Kako raditi s ovom bilježnicom

> **Prvo spremi svoju kopiju:** `File → Save Notebook As…` → `EX_5_moj_rad.ipynb` i radi u njoj.
> Tako `git pull` nikad neće javiti sukob s tvojim izmjenama.

Svaka cjelina ima isti raspored:

| Oznaka | Što znači |
|---|---|
| 📖 **Teorija** | kratak podsjetnik; broj slajda odnosi se na **1. prezentaciju s predavanja** |
| ▶️ **Demo** | kod koji pokrećemo zajedno (`Shift + Enter`) |
| 🧩 **Zadatak** | ćelija s oznakom `# TODO` – zamijeni `...` ili `___` svojim kodom |
| ✅ **Provjera** | pokreni odmah nakon zadatka; ispisat će ✅ ili objasniti što ne valja (provjere su u datoteci `provjere.py`) |
| ✏️ | odgovor upisuješ u markdown ćeliju (dvoklik na ćeliju, zatim `Shift + Enter`) |
| ⭐ | dodatni zadatak – ako ti ostane vremena na satu, ili za doma |

Ćelije pokreći **redom, odozgo prema dolje**. SQL još ne moramo znati: upiti su pripremljeni, a ti popunjavaš samo označena mjesta.
Ako zapneš na zadatku, nastavi dalje. Cjeline 4–8 kreću od gotove baze, pa rade i ako prethodni zadatak nije riješen.

# Sadržaj

Radimo na primjeru maloprodajnog lanca **MarketPlus** iz **radnog lista 3** (Star vs. Snowflake) i pretvaramo ga u pravo malo skladište podataka.

| # | Cjelina | ≈ min |
|---|---|---|
| 0 | Priprema okruženja | 4 |
| 1 | SQLite u dvije minute i sirovi podaci | 6 |
| 2 | Pet koraka dimenzijskog modeliranja | 10 |
| 3 | Gradnja star sheme | 20 |
| 4 | Analitički upiti nad star shemom | 10 |
| 5 | Snowflake shema | 8 |
| 6 | Zamka granulacije | 7 |
| 7 | Sporo mijenjajuće dimenzije (SCD) | 25 |
| 8 | ⭐ Završni zadatak: SCD 2 za proizvod | doma |
| 9 | Neobavezno: veza sa SQL podom | – |

Svi podaci su **izmišljeni** za potrebe vježbe. Prva četiri retka sirovih podataka iste su prodaje kao u radnom listu 3.

# 0. Priprema okruženja (≈ 4 min)

Radimo u Jupyter podu (`okruzenje/anaconda_jupyter.yaml`) i ne treba nam dodatni pod: **SQLite** dolazi s Pythonom.

In [ ]:
# Samo ako je nastavnik najavio ispravak materijala – povlači najnoviju verziju repozitorija.
# (Radiš u svojoj kopiji, pa se tvoje izmjene neće prebrisati.)
!git pull

In [ ]:
import importlib
import os
import shutil
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import provjere
importlib.reload(provjere)        # nakon git pull učitava novu verziju provjera

pd.set_option("display.max_columns", 30)
print("pandas", pd.__version__, "| SQLite", sqlite3.sqlite_version)

In [ ]:
# Naša baza je jedna datoteka. Ćelija kreće od prazne baze, pa je možeš pokrenuti koliko god puta želiš.
BAZA = "marketplus.db"
if "conn" in globals():
    conn.close()
if os.path.exists(BAZA):
    os.remove(BAZA)

conn = sqlite3.connect(BAZA)
conn.execute("PRAGMA foreign_keys = ON")     # SQLite provjerava strane ključeve samo ako to uključimo


def sql(upit):
    """Izvrši SELECT upit i vrati rezultat kao pandas DataFrame."""
    return pd.read_sql_query(upit, conn)


def upisi(df, tablica):
    """Zamijeni sadržaj tablice retcima iz DataFramea."""
    conn.commit()
    conn.execute("PRAGMA foreign_keys = OFF")
    conn.execute(f"DELETE FROM {tablica}")
    df.to_sql(tablica, conn, if_exists="append", index=False)
    conn.commit()
    conn.execute("PRAGMA foreign_keys = ON")


def otvori_gotovu_bazu(izvor="podaci/marketplus_gotovo.db"):
    """Otvori svježu kopiju provjerene baze, da svi nastavimo od istog stanja."""
    global conn
    conn.close()
    shutil.copy(izvor, "marketplus_rad.db")
    conn = sqlite3.connect("marketplus_rad.db")
    conn.execute("PRAGMA foreign_keys = ON")

# 1. SQLite u dvije minute i sirovi podaci (≈ 6 min)

📖 **Teorija** – SQLite je prava relacijska baza (tablice, ključevi, SQL) koja živi u **jednoj datoteci**, bez poslužitelja.
Za produkcijsko skladište koristili bismo npr. PostgreSQL ili Snowflake, ali **model je isti**, a to nas danas zanima.

Sve što nam treba iz SQL-a stane u nekoliko riječi:

| SQL | Što radi | U pandasu |
|---|---|---|
| `SELECT stupci` | koje stupce želimo | `df[[...]]` |
| `FROM tablica` | iz koje tablice | |
| `JOIN druga ON a = b` | spoji dvije tablice po ključu | `merge` |
| `WHERE uvjet` | filtriraj retke | `df[uvjet]` |
| `GROUP BY stupac` | grupiraj | `groupby` |
| `SUM(stupac)` | zbroj unutar grupe | `.sum()` |
| `ORDER BY stupac` | poredaj | `sort_values` |

▶️ **Demo** – sirovi podaci o prodaji, onakvi kakve dobivamo iz blagajni:

In [ ]:
prodaja = pd.read_csv("podaci/prodaja_sirovo.csv")
print(prodaja.shape)
prodaja.head(6)

In [ ]:
# Ista tablica u SQLite-u i prvi upit: prihod po kategoriji
prodaja.to_sql("sirovo", conn, if_exists="replace", index=False)

sql("""
    SELECT kategorija, SUM(iznos) AS prihod
    FROM sirovo
    GROUP BY kategorija
    ORDER BY prihod DESC
""")

Sve je u jednoj širokoj tablici. Pogledaj koliko se puta ponavlja isti grad trgovine ili ime kupca.
Promjena naziva jedne kategorije značila bi izmjenu stotina redaka (anomalija ažuriranja iz normalizacije).
Za analitiku zato gradimo **dimenzijski model**.

# 2. Pet koraka dimenzijskog modeliranja (≈ 10 min)

📖 **Teorija** (sl. 137–139)

1. **Poslovni proces** – što mjerimo? Ovdje: prodaja u trgovini.
2. **Granularnost (grain)** – što točno predstavlja **jedan redak** tablice činjenica? To je najvažnija odluka: sve ostalo slijedi iz nje.
3. **Dimenzije** – kontekst: tko, što, gdje, kada.
4. **Činjenice (mjere)** – brojevi koje zbrajamo.
5. **Izgradnja sheme.**

🧩 **Zadatak 1** – odredi granularnost i mjere tablice `prodaja`.

In [ ]:
# TODO a) Koji stupci zajedno jednoznačno određuju jedan redak? (popis naziva stupaca)
stupci_granulacije = ...

# TODO b) Koji su stupci MJERE na toj granularnosti? Zbrajaju se po stavkama, a svaka stavka ima svoju vrijednost.
#         Pomoć: usporedi stupce kolicina, iznos i popust_racuna za neki račun s više stavki (npr. R00038).
mjere = ...

prodaja[prodaja["broj_racuna"] == "R00038"][["broj_racuna", "stavka", "proizvod", "kolicina", "iznos", "popust_racuna"]]

In [ ]:
# ✅ Provjera
provjere.zadatak_1(prodaja, stupci_granulacije, mjere)

✏️ **Zadatak 1 c)** Dovrši model (kao u radnom listu 3). Granularnost napiši jednom rečenicom: *„jedan redak = …”*.

**Granularnost:**

| Dimenzija | Poslovni ključ iz izvora | Atributi |
|---|---|---|
| `dim_datum` | `datum` | |
| `dim_trgovina` | | |
| `dim_proizvod` | | |
| `dim_kupac` | | |

# 3. Gradnja star sheme (≈ 20 min)

📖 **Teorija** (sl. 141–142, 148) – u **star shemi** tablica činjenica je u sredini, a svaka dimenzija povezana je s njom izravno.

Svaka dimenzija ima dva ključa:
* **poslovni (prirodni) ključ** dolazi iz izvora (`sifra_kupca`),
* **surogatni ključ** dodjeljujemo sami u skladištu (`kupac_key`): običan broj bez poslovnog značenja. Tablica činjenica koristi njega. U cjelini 7 vidjet ćemo zašto.

▶️ **Demo** – stvaramo prazne tablice. Pogledaj `REFERENCES`: to su strani ključevi s predavanja.

In [ ]:
conn.executescript("""
    DROP TABLE IF EXISTS fact_prodaja;
    DROP TABLE IF EXISTS dim_datum;
    DROP TABLE IF EXISTS dim_trgovina;
    DROP TABLE IF EXISTS dim_proizvod;
    DROP TABLE IF EXISTS dim_kupac;

CREATE TABLE dim_datum (
    datum_key     INTEGER PRIMARY KEY,   -- npr. 20260305
    datum         TEXT NOT NULL,
    godina        INTEGER NOT NULL,
    kvartal       INTEGER NOT NULL,
    mjesec        INTEGER NOT NULL,
    naziv_mjeseca TEXT NOT NULL,
    dan_u_tjednu  INTEGER NOT NULL,      -- 1 = ponedjeljak
    vikend        INTEGER NOT NULL
);
CREATE TABLE dim_trgovina (
    trgovina_key   INTEGER PRIMARY KEY,  -- surogatni ključ
    sifra_trgovine TEXT NOT NULL,        -- poslovni ključ iz izvora
    naziv          TEXT NOT NULL,
    grad           TEXT NOT NULL,
    regija         TEXT NOT NULL
);
CREATE TABLE dim_proizvod (
    proizvod_key    INTEGER PRIMARY KEY,
    sifra_proizvoda TEXT NOT NULL,
    naziv           TEXT NOT NULL,
    kategorija      TEXT NOT NULL
);
CREATE TABLE dim_kupac (
    kupac_key   INTEGER PRIMARY KEY,
    sifra_kupca TEXT NOT NULL,
    kupac       TEXT NOT NULL,
    grad        TEXT NOT NULL
);

CREATE TABLE fact_prodaja (
    prodaja_key  INTEGER PRIMARY KEY,
    datum_key    INTEGER NOT NULL REFERENCES dim_datum(datum_key),
    trgovina_key INTEGER NOT NULL REFERENCES dim_trgovina(trgovina_key),
    proizvod_key INTEGER NOT NULL REFERENCES dim_proizvod(proizvod_key),
    kupac_key    INTEGER NOT NULL REFERENCES dim_kupac(kupac_key),
    broj_racuna  TEXT NOT NULL,          -- ostaje u fact tablici: vraća nas na izvorni račun
    stavka       INTEGER NOT NULL,
    kolicina     INTEGER NOT NULL,       -- mjera
    iznos        REAL NOT NULL           -- mjera
);
""")
sql("SELECT name AS tablica FROM sqlite_master WHERE type = 'table' ORDER BY name")

▶️ **Demo** – punimo dimenziju datuma i dvije dimenzije iz sirovih podataka.
`dim_datum` ne dolazi iz izvora: sami je generiramo za cijelu godinu, pa imamo i dane bez prodaje.

In [ ]:
MJESECI = ["siječanj", "veljača", "ožujak", "travanj", "svibanj", "lipanj", "srpanj", "kolovoz", "rujan", "listopad", "studeni", "prosinac"]
dani = pd.date_range("2026-01-01", "2026-12-31")

dim_datum_df = pd.DataFrame({
    "datum_key": dani.strftime("%Y%m%d").astype(int),
    "datum": dani.strftime("%Y-%m-%d"),
    "godina": dani.year,
    "kvartal": dani.quarter,
    "mjesec": dani.month,
    "naziv_mjeseca": [MJESECI[m - 1] for m in dani.month],
    "dan_u_tjednu": dani.dayofweek + 1,
    "vikend": (dani.dayofweek >= 5).astype(int),
})
upisi(dim_datum_df, "dim_datum")
dim_datum_df.iloc[63:67]

In [ ]:
# dim_trgovina: jedinstvene trgovine + surogatni ključ 1, 2, 3 …
dim_trgovina_df = (
    prodaja[["sifra_trgovine", "trgovina", "grad_trgovine", "regija"]]
    .drop_duplicates()
    .sort_values("sifra_trgovine")
    .rename(columns={"trgovina": "naziv", "grad_trgovine": "grad"})
    .reset_index(drop=True)
)
dim_trgovina_df.insert(0, "trgovina_key", range(1, len(dim_trgovina_df) + 1))
upisi(dim_trgovina_df, "dim_trgovina")

# dim_proizvod: isti postupak
dim_proizvod_df = (
    prodaja[["sifra_proizvoda", "proizvod", "kategorija"]]
    .drop_duplicates()
    .sort_values("sifra_proizvoda")
    .rename(columns={"proizvod": "naziv"})
    .reset_index(drop=True)
)
dim_proizvod_df.insert(0, "proizvod_key", range(1, len(dim_proizvod_df) + 1))
upisi(dim_proizvod_df, "dim_proizvod")

dim_trgovina_df

🧩 **Zadatak 2** – po uzoru na `dim_trgovina` izgradi `dim_kupac_df` sa stupcima `kupac_key`, `sifra_kupca`, `kupac`, `grad`.

In [ ]:
# TODO Jedinstveni kupci, poredani po sifra_kupca, sa stupcem grad_kupca preimenovanim u grad.
dim_kupac_df = ...

# TODO Dodaj surogatni ključ kupac_key (1, 2, 3 …) kao prvi stupac: dim_kupac_df.insert(...)
...

In [ ]:
# ✅ Provjera (ako je točno, dimenzija se upisuje u bazu)
provjere.zadatak_2(dim_kupac_df)
upisi(dim_kupac_df, "dim_kupac")

🧩 **Zadatak 3** – tablica činjenica. Svaki poslovni ključ zamjenjujemo surogatnim ključem iz dimenzije: to je operacija `merge` (u SQL-u `JOIN`).
Za `datum` i trgovinu to je već napravljeno.

In [ ]:
f = prodaja.copy()
f["datum_key"] = pd.to_datetime(f["datum"]).dt.strftime("%Y%m%d").astype(int)
f = f.merge(dim_trgovina_df[["sifra_trgovine", "trgovina_key"]], on="sifra_trgovine")

# TODO Na isti način dodaj proizvod_key (preko sifra_proizvoda) i kupac_key (preko sifra_kupca).
f = ...
f = ...

In [ ]:
# ✅ Provjera (ako je točno, tablica činjenica se upisuje u bazu)
provjere.zadatak_3(f, prodaja)
fact_df = f[["datum_key", "trgovina_key", "proizvod_key", "kupac_key", "broj_racuna", "stavka", "kolicina", "iznos"]]
upisi(fact_df, "fact_prodaja")
sql("SELECT * FROM fact_prodaja LIMIT 5")

▶️ **Demo** – strani ključ u akciji: baza odbija redak koji pokazuje na nepostojeću trgovinu.

In [ ]:
try:
    conn.execute("""
        INSERT INTO fact_prodaja (datum_key, trgovina_key, proizvod_key, kupac_key, broj_racuna, stavka, kolicina, iznos)
        VALUES (20260305, 999, 1, 1, 'TEST', 1, 1, 1.0)
    """)
except sqlite3.IntegrityError as greska:
    print("Baza je odbila redak:", greska)
conn.rollback()

# 4. Analitički upiti nad star shemom (≈ 10 min)

Od ove točke svi radimo na **provjerenoj kopiji** baze, pa kasnije cjeline rade i ako zadaci 2–3 nisu dovršeni.

▶️ **Demo** – svaki upit ima isti oblik: krenemo od `fact_prodaja`, spojimo dimenzije koje trebamo, grupiramo po njihovim atributima i zbrojimo mjeru.
Slova `f`, `p` i `d` su kratka imena (aliasi) za tablice.

In [ ]:
otvori_gotovu_bazu()

sql("""
    SELECT p.kategorija, d.naziv_mjeseca, SUM(f.iznos) AS prihod
    FROM fact_prodaja f
    JOIN dim_proizvod p ON f.proizvod_key = p.proizvod_key
    JOIN dim_datum d    ON f.datum_key = d.datum_key
    GROUP BY p.kategorija, d.mjesec, d.naziv_mjeseca
    ORDER BY d.mjesec, prihod DESC
""")

In [ ]:
# Rezultat upita je običan DataFrame, pa ga možemo odmah nacrtati.
po_danu = sql("""
    SELECT d.dan_u_tjednu, SUM(f.iznos) AS prihod
    FROM fact_prodaja f
    JOIN dim_datum d ON f.datum_key = d.datum_key
    GROUP BY d.dan_u_tjednu
    ORDER BY d.dan_u_tjednu
""")
po_danu.plot(kind="bar", x="dan_u_tjednu", y="prihod", legend=False, rot=0,
             title="Prihod po danu u tjednu (1 = ponedjeljak)", ylabel="prihod (€)", xlabel="", figsize=(7, 3))
plt.show()

🧩 **Zadatak 4** – popuni mjesta označena s `___`. Upit treba dati **prihod po mjesecu i gradu trgovine**, sa stupcima `naziv_mjeseca`, `grad`, `prihod`.

In [ ]:
# TODO Popuni ___ (mjera koju zbrajamo i ključevi po kojima spajamo trgovinu)
upit_mjesec_grad = """
    SELECT d.naziv_mjeseca, t.grad, SUM(f.___) AS prihod
    FROM fact_prodaja f
    JOIN dim_datum d    ON f.datum_key = d.datum_key
    JOIN dim_trgovina t ON f.___ = t.___
    GROUP BY d.mjesec, d.naziv_mjeseca, t.grad
    ORDER BY d.mjesec, prihod DESC
"""

In [ ]:
# ✅ Provjera
provjere.zadatak_4(sql, upit_mjesec_grad)

# 5. Snowflake shema (≈ 8 min)

📖 **Teorija** (sl. 143–147) – u **snowflake shemi** dimenzije su normalizirane: dio atributa izdvajamo u pod-dimenziju.
U `dim_trgovina` regija ovisi o gradu, a ne o trgovini (tranzitivna ovisnost iz 3NF). Zato izdvajamo `dim_grad`, kao u radnom listu 3.

▶️ **Demo**

In [ ]:
conn.executescript("""
    DROP TABLE IF EXISTS dim_trgovina_sf;
    DROP TABLE IF EXISTS dim_grad;

    CREATE TABLE dim_grad (
        grad_key INTEGER PRIMARY KEY,
        grad     TEXT NOT NULL,
        regija   TEXT NOT NULL
    );
    INSERT INTO dim_grad (grad, regija)
    SELECT DISTINCT grad, regija FROM dim_trgovina ORDER BY grad;

    CREATE TABLE dim_trgovina_sf (
        trgovina_key   INTEGER PRIMARY KEY,
        sifra_trgovine TEXT NOT NULL,
        naziv          TEXT NOT NULL,
        grad_key       INTEGER NOT NULL REFERENCES dim_grad(grad_key)
    );
    INSERT INTO dim_trgovina_sf
    SELECT t.trgovina_key, t.sifra_trgovine, t.naziv, g.grad_key
    FROM dim_trgovina t
    JOIN dim_grad g ON t.grad = g.grad;
""")
display(sql("SELECT * FROM dim_grad"))
display(sql("SELECT * FROM dim_trgovina_sf"))

🧩 **Zadatak 5** – isti izvještaj (**prihod po regiji**) sada traži spajanje preko dodatne tablice. Popuni `___`.

In [ ]:
# TODO Spoji dim_trgovina_sf s dim_grad
upit_regija = """
    SELECT g.regija, SUM(f.iznos) AS prihod
    FROM fact_prodaja f
    JOIN dim_trgovina_sf t ON f.trgovina_key = t.trgovina_key
    JOIN ___ g ON t.___ = g.___
    GROUP BY g.regija
    ORDER BY prihod DESC
"""

In [ ]:
# ✅ Provjera
provjere.zadatak_5(sql, upit_regija)

✏️ **Zadatak 5 b)** Usporedi obje sheme (tablica iz radnog lista 3). Odgovori na temelju onoga što si upravo napravio/la.

| Kriterij | Star shema | Snowflake shema |
|---|---|---|
| Broj spajanja za izvještaj po regiji | | |
| Gdje se ponavlja naziv regije | | |
| Što treba promijeniti ako se regija preimenuje | | |
| Razumljivost za poslovnog korisnika | | |

# 6. Zamka granulacije (≈ 7 min)

📖 **Teorija** – stupac `popust_racuna` je popust na **cijeli račun**, a izvor ga ponavlja u svakoj stavci.
Ta vrijednost nije na istoj granularnosti kao redak (jedna stavka), pa je `SUM` daje krivi rezultat. To je najčešća greška u analitici: brojevi izgledaju uvjerljivo, a pogrešni su.

▶️ **Demo**

In [ ]:
print("Naivni zbroj popusta:", round(prodaja["popust_racuna"].sum(), 2), "€")
prodaja[prodaja["popust_racuna"] > 0].groupby("broj_racuna").filter(lambda g: len(g) > 1)[
    ["broj_racuna", "stavka", "proizvod", "iznos", "popust_racuna"]].head(5)

🧩 **Zadatak 6** – izračunaj stvarni ukupni popust, tako da svaki račun brojiš **samo jednom**.
Pomoć: `drop_duplicates(subset=...)`.

In [ ]:
# TODO Stvarni ukupni popust (svaki račun jednom)
stvarni_popust = ...

In [ ]:
# ✅ Provjera
provjere.zadatak_6(stvarni_popust)

▶️ **Demo** – kako bismo popust ispravno spremili? Dvije su mogućnosti:
* zasebna tablica činjenica na granularnosti **računa**, ili
* **raspodjela** popusta po stavkama, razmjerno njihovom iznosu, kao ovdje. Tada se popust smije zbrajati.

In [ ]:
raspodjela = prodaja.copy()
raspodjela["iznos_racuna"] = raspodjela.groupby("broj_racuna")["iznos"].transform("sum")
raspodjela["popust_stavke"] = (raspodjela["popust_racuna"] * raspodjela["iznos"] / raspodjela["iznos_racuna"]).round(4)

raspodjela[raspodjela["broj_racuna"] == "R00038"][["broj_racuna", "proizvod", "iznos", "popust_racuna", "popust_stavke"]]

# 7. Sporo mijenjajuće dimenzije – SCD (≈ 25 min)

📖 **Teorija** (sl. 150) – atributi u dimenzijama mijenjaju se rijetko, ali se mijenjaju: kupac se preseli, proizvod promijeni kategoriju.
Kako to spremiti, a da ne pokvarimo povijesne izvještaje? Takve dimenzije zovemo **sporo mijenjajuće dimenzije** (*slowly changing dimensions*). Tri su glavne politike:

| Politika s predavanja | Naziv | Što radimo | Povijest |
|---|---|---|---|
| Prepisivanje | **SCD 1** | `UPDATE` postojećeg retka | gubi se |
| Održavanje neograničene povijesti | **SCD 2** | novi redak za svaku verziju | čuva se u cijelosti |
| Održavanje ograničene povijesti | **SCD 3** | dodatni stupac s prethodnom vrijednošću | samo jedna prethodna vrijednost |

**Scenarij:** kupac `K001` (A. Kovač) živio je u Zagrebu, a **1. 6. 2026.** preselio se u Split. Kupac `K003` selio se sredinom srpnja.

In [ ]:
promjene = pd.read_csv("podaci/promjene_kupaca.csv")
promjene

▶️ **Demo – SCD 1 (prepisivanje).** Izvještaj pokriva prodaju **do kraja svibnja**, kad je A. Kovač još živio u Zagrebu.

In [ ]:
otvori_gotovu_bazu()

def izvjestaj_po_gradu():
    return sql("""
        SELECT k.grad AS grad_kupca, ROUND(SUM(f.iznos), 2) AS prihod
        FROM fact_prodaja f
        JOIN dim_kupac k ON f.kupac_key = k.kupac_key
        GROUP BY k.grad
    """).set_index("grad_kupca")["prihod"]

prije = izvjestaj_po_gradu()
conn.execute("UPDATE dim_kupac SET grad = 'Split' WHERE sifra_kupca = 'K001'")   # SCD 1
poslije = izvjestaj_po_gradu()
conn.rollback()                                                                  # poništavamo izmjenu

pd.DataFrame({"prije": prije, "poslije SCD 1": poslije, "razlika": (poslije - prije).round(2)}).query("razlika != 0")

**Prošlost se promijenila**, iako nitko nije ni kupio ni vratio ništa: prodaja iz ožujka odjednom pripada Splitu.
Izvještaj predan upravi u svibnju više se ne može ponoviti. SCD 1 je dobar samo za ispravke grešaka (npr. tipfeler u imenu).

▶️ **Demo – SCD 3 (ograničena povijest).** Dodajemo stupac `prethodni_grad`.

In [ ]:
otvori_gotovu_bazu()

conn.execute("ALTER TABLE dim_kupac ADD COLUMN prethodni_grad TEXT")
conn.execute("UPDATE dim_kupac SET prethodni_grad = grad, grad = 'Split' WHERE sifra_kupca = 'K001'")
conn.commit()
display(sql("SELECT * FROM dim_kupac WHERE sifra_kupca = 'K001'"))

Znamo prethodni grad, ali ne i **kada** se promijenio, a druga selidba bi izbrisala Zagreb. Stara prodaja i dalje se računa u Split.

📖 **SCD 2 (neograničena povijest).** Za svaku verziju kupca postoji zaseban redak sa svojim **surogatnim ključem**:

| kupac_key | sifra_kupca | grad | valid_from | valid_to | is_current |
|---|---|---|---|---|---|
| 1 | K001 | Zagreb | 1900-01-01 | 2026-05-31 | 0 |
| 21 | K001 | Split | 2026-06-01 | 9999-12-31 | 1 |

Sad je jasno zašto trebamo surogatni ključ: `sifra_kupca` se ponavlja, a `kupac_key` je i dalje jedinstven.
Stara prodaja i dalje pokazuje na ključ 1 (Zagreb), a nova dobiva ključ 21 (Split). **Tablicu činjenica ne diramo.**

▶️ **Demo** – pripremamo dimenziju: dodajemo tri nova stupca. `9999-12-31` je dogovor koji znači „vrijedi i dalje”.

In [ ]:
otvori_gotovu_bazu()

conn.executescript("""
    ALTER TABLE dim_kupac ADD COLUMN valid_from TEXT;
    ALTER TABLE dim_kupac ADD COLUMN valid_to TEXT;
    ALTER TABLE dim_kupac ADD COLUMN is_current INTEGER;
    UPDATE dim_kupac SET valid_from = '1900-01-01', valid_to = '9999-12-31', is_current = 1;
""")
sql("SELECT * FROM dim_kupac LIMIT 3")

🧩 **Zadatak 7** – dovrši funkciju `primijeni_scd2`. Radi dvije stvari: **zatvara** trenutni redak kupca i **dodaje** novi.
Upitnici `?` zamjenjuju se vrijednostima iz zagrada, redom. `date(?, '-1 day')` je dan prije datuma promjene.

In [ ]:
def primijeni_scd2(conn, sifra_kupca, novi_grad, datum_promjene):
    # TODO 1) Zatvori trenutni redak: stari redak više nije važeći (popuni ___).
    conn.execute("""
        UPDATE dim_kupac
        SET valid_to = date(?, '-1 day'),
            is_current = ___
        WHERE sifra_kupca = ? AND is_current = 1
    """, (datum_promjene, sifra_kupca))

    # TODO 2) Dodaj novi redak: vrijedi od datuma promjene i trenutno je važeći (popuni dva ___).
    conn.execute("""
        INSERT INTO dim_kupac (sifra_kupca, kupac, grad, valid_from, valid_to, is_current)
        SELECT sifra_kupca, kupac, ?, ?, '___', ___
        FROM dim_kupac
        WHERE sifra_kupca = ? AND valid_to = date(?, '-1 day')
    """, (novi_grad, datum_promjene, sifra_kupca, datum_promjene))
    conn.commit()

In [ ]:
# ✅ Provjera (radi na kopiji baze, pa je možeš pokretati koliko god puta želiš)
provjere.zadatak_7(conn, primijeni_scd2, promjene)

🧩 **Zadatak 8** – stigla je prodaja za lipanj i srpanj. Svaka stavka mora dobiti `kupac_key` **verzije koja je vrijedila na datum prodaje**.
`dim_kupac_scd2.csv` je gotova dimenzija nakon obje selidbe. Datume smijemo uspoređivati kao tekst, jer su u obliku `GGGG-MM-DD`.

In [ ]:
nova_prodaja = pd.read_csv("podaci/nova_prodaja.csv")
dim_scd2 = pd.read_csv("podaci/dim_kupac_scd2.csv")

# merge po sifra_kupca daje po jedan redak za SVAKU verziju kupca
spojeno = nova_prodaja.merge(dim_scd2[["kupac_key", "sifra_kupca", "valid_from", "valid_to"]], on="sifra_kupca")

# TODO Zadrži samo retke u kojima je datum prodaje između valid_from i valid_to (uključivo).
nova_s_kljucem = ...

In [ ]:
# ✅ Provjera
provjere.zadatak_8(nova_prodaja, nova_s_kljucem)

▶️ **Demo** – nagrada za trud. Otvaramo bazu u kojoj je dimenzija po SCD 2 i učitana je i nova prodaja. Sad možemo odgovoriti na **dva različita poslovna pitanja**:

In [ ]:
otvori_gotovu_bazu("podaci/marketplus_scd2_gotovo.db")

# Gdje je kupac živio U TRENUTKU kupnje? Tablicu spajamo izravno po kupac_key.
u_trenutku = sql("""
    SELECT k.grad, ROUND(SUM(f.iznos), 2) AS prihod
    FROM fact_prodaja f
    JOIN dim_kupac k ON f.kupac_key = k.kupac_key
    GROUP BY k.grad
""").set_index("grad")["prihod"]

# Gdje kupac živi DANAS? Preko poslovnog ključa dolazimo do trenutne verzije.
danas = sql("""
    SELECT sad.grad, ROUND(SUM(f.iznos), 2) AS prihod
    FROM fact_prodaja f
    JOIN dim_kupac k   ON f.kupac_key = k.kupac_key
    JOIN dim_kupac sad ON k.sifra_kupca = sad.sifra_kupca AND sad.is_current = 1
    GROUP BY sad.grad
""").set_index("grad")["prihod"]

pd.DataFrame({"grad u trenutku kupnje": u_trenutku, "današnji grad kupca": danas}).sort_values("današnji grad kupca", ascending=False)

In [ ]:
# Povijest kupca K001 kroz vrijeme: isti poslovni ključ, dva surogatna ključa
sql("""
    SELECT k.kupac_key, k.sifra_kupca, k.grad, k.valid_from, k.valid_to, k.is_current,
           COUNT(*) AS stavki, ROUND(SUM(f.iznos), 2) AS prihod
    FROM fact_prodaja f
    JOIN dim_kupac k ON f.kupac_key = k.kupac_key
    WHERE k.sifra_kupca = 'K001'
    GROUP BY k.kupac_key
""")

✏️ **Zadatak 9** – koju bi politiku odabrao/la za svaku promjenu? Obrazloži jednom rečenicom.

| Promjena | SCD 1 / 2 / 3 | Zašto? |
|---|---|---|
| Ispravak tipfelera u imenu kupca (`A. Kovac` → `A. Kovač`) | | |
| Kupac se preselio u drugi grad | | |
| Proizvod je prebačen u drugu kategoriju, a uprava želi usporedbe prije i poslije | | |
| Uprava želi jedan izvještaj „po staroj i novoj organizaciji regija”, samo za ovu godinu | | |

# 8. ⭐ Završni zadatak: SCD 2 za proizvod (doma)

`Sok naranča 1L` (`P003`) od **1. 6. 2026.** više nije u kategoriji `Prehrana`, nego u novoj kategoriji `Pića`.
Napiši sve sam/a, po uzoru na zadatke 7 i 8. Ćelija s provjerom ispisuje rezultat za svaki korak.

In [ ]:
otvori_gotovu_bazu("podaci/marketplus_scd2_gotovo.db")

conn.executescript("""
    ALTER TABLE dim_proizvod ADD COLUMN valid_from TEXT;
    ALTER TABLE dim_proizvod ADD COLUMN valid_to TEXT;
    ALTER TABLE dim_proizvod ADD COLUMN is_current INTEGER;
    UPDATE dim_proizvod SET valid_from = '1900-01-01', valid_to = '9999-12-31', is_current = 1;
""")
promjene_proizvoda = pd.read_csv("podaci/promjene_proizvoda.csv")
promjene_proizvoda

In [ ]:
def primijeni_scd2_proizvod(conn, sifra_proizvoda, nova_kategorija, datum_promjene):
    # TODO Zatvori trenutni redak proizvoda i dodaj novi.
    ...


# TODO 2) Primijeni promjenu iz tablice promjene_proizvoda.
...

# TODO 3) Pročitaj dimenziju u DataFrame i za nova_prodaja pronađi proizvod_key koji je vrijedio na datum prodaje.
#         Rezultat spremi u nova_s_proizvodom; mora imati isti broj redaka kao nova_prodaja i stupac proizvod_key.
nova_prodaja = pd.read_csv("podaci/nova_prodaja.csv")
nova_s_proizvodom = ...

In [ ]:
# ✅ Provjera – svaki korak zasebno
provjere.zadatak_10(sql, nova_prodaja, nova_s_proizvodom)

⭐⭐ **Za one koji žele više**
* Novi kupac `K021` pojavio se tek u kolovozu. Što mora napraviti učitavanje da mu `merge` ne izgubi retke?
* Napiši jednu opću funkciju `scd2(conn, tablica, kljuc, stupac, nova_vrijednost, datum)` koja radi i za kupce i za proizvode.
* Što ako promjena stigne s kašnjenjem, nakon što je prodaja već učitana sa starim ključem?

# 9. Neobavezno: veza sa SQL podom

**Ovo je samo za one koji su već pokrenuli SQL pod** (`postgres-pgadmin-pod`). Ostali mogu preskočiti ovu cjelinu.
Upiti samo čitaju podatke i ne mijenjaju ništa u bazi.

Pogledaj što smo upravo gradili u pravom PostgreSQL skladištu: `dim_customer` ima iste stupce `valid_from`, `valid_to` i `is_current`.

> Iz Jupyter poda baza je dostupna na adresi `postgres-pgadmin-pod:5432`.
> Adresa `localhost:5433` vrijedi samo s tvojeg računala (npr. za pgAdmin ili DBeaver), a ne iz spremnika.

In [ ]:
%pip install -q psycopg2-binary

In [ ]:
try:
    import psycopg2

    veza = psycopg2.connect(host="postgres-pgadmin-pod", port=5432, dbname="dwh",
                            user="myuser", password="mypassword", connect_timeout=3)
except Exception as greska:
    print("Veza nije uspjela – je li SQL pod pokrenut? Ako ga nemaš, preskoči ovu cjelinu.")
    print("Detalj:", str(greska).strip()[:150])
else:
    with veza.cursor() as kursor:
        kursor.execute("""
            SELECT column_name, data_type
            FROM information_schema.columns
            WHERE table_name = 'dim_customer'
            ORDER BY ordinal_position
        """)
        display(pd.DataFrame(kursor.fetchall(), columns=["stupac", "tip"]))
    veza.close()
    print("✅ Veza sa SQL podom radi.")

# Sažetak

* **Granularnost** odredi prvu: jedan redak tablice činjenica mora značiti jednu stvar. Vrijednost s druge razine (popust na račun) ne smije se samo zbrojiti.
* **Star shema:** tablica činjenica sa surogatnim ključevima i mjerama te denormalizirane dimenzije. Upit je uvijek isti: `fact` + `JOIN` dimenzija + `GROUP BY`.
* **Snowflake** normalizira dimenzije: manje ponavljanja, ali više spajanja.
* **SCD 1** prepisuje i gubi povijest, **SCD 3** čuva jednu prethodnu vrijednost, a **SCD 2** dodaje redak za svaku verziju.
* Kod SCD 2 tablicu činjenica ne mijenjamo: nova prodaja dobiva ključ verzije koja je vrijedila na datum prodaje. Zato trebamo **surogatni ključ**.

**Izvori:** 1. prezentacija s predavanja (DWH, sl. 137–150) i radni list 3 *Star vs. Snowflake*. Podaci su izmišljeni za potrebe vježbe.